In [1]:
import numpy as np
import cvxpy as cp
import scipy.linalg as la
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches


# --- Configuração dos Parâmetros Físicos e Hiperparâmetros ---
alpha_step = 0.1       # Taxa de integração/vazamento
n_x = 2                # Dimensão do Estado [theta, theta_dot]
n_u = 1                # Dimensão do Controle [u]
n_h = 64               # Neurónios por camada oculta
n_layers = 3           # Número de camadas ocultas não-lineares
N_total = n_h * n_layers # Total de neurónios (192)

# Limites Físicos (Para validação posterior)
x_max = np.array([0.573, 3.0]) # [rad, rad/s]
u_max = 0.7                    # [Nm]

# --- 2. Carregamento dos Pesos da Rede Neural ---
# Caminho da pasta contendo os arquivos CSV
data_path = 'matrizes_lure_3x64'

def load_weight_matrix(filename):
    """
    Carrega uma matriz CSV sem cabeçalho e retorna como array numpy.
    """
    full_path = os.path.join(data_path, filename)
    try:
        # header=None assume que o CSV contém apenas números
        df = pd.read_csv(full_path, header=None)
        return df.values
    except FileNotFoundError:
        print(f"ERRO: O ficheiro '{filename}' não foi encontrado em '{data_path}'.")
        return None

print("--- Iniciando Carregamento dos Pesos ---")

# Mapeamento para as variáveis matemáticas do projeto
# W_in: Camada de Entrada -> W_in
W_in = load_weight_matrix('W_in.csv')

# W_h1: Hidden 1 -> Hidden 2 (Arquivo W_hidden_1)
W_h1 = load_weight_matrix('W_hidden_1.csv')

# W_h2: Hidden 2 -> Hidden 3 (Arquivo W_hidden_2)
W_h2 = load_weight_matrix('W_hidden_2.csv')

# W_out: Hidden 3 -> Saída (Arquivo W_out)
W_out = load_weight_matrix('W_out.csv')

# W_u: Controle -> Estado (Arquivo W_u)
W_u = load_weight_matrix('W_u.csv')

# --- 3. Validação das Dimensões ---
if all(w is not None for w in [W_in, W_h1, W_h2, W_out, W_u]):
    print("\nTodas as matrizes foram carregadas.")
    print(f"W_in shape:  {W_in.shape}  (Esperado: {n_h}x{n_x})")
    print(f"W_h1 shape:  {W_h1.shape}  (Esperado: {n_h}x{n_h})")
    print(f"W_h2 shape:  {W_h2.shape}  (Esperado: {n_h}x{n_h})")
    print(f"W_out shape: {W_out.shape} (Esperado: {n_x}x{n_h})")
    print(f"W_u shape:   {W_u.shape}   (Esperado: {n_x}x{n_u})")
else:
    print("\nFALHA")

--- Iniciando Carregamento dos Pesos ---

Todas as matrizes foram carregadas.
W_in shape:  (64, 2)  (Esperado: 64x2)
W_h1 shape:  (64, 64)  (Esperado: 64x64)
W_h2 shape:  (64, 64)  (Esperado: 64x64)
W_out shape: (2, 64) (Esperado: 2x64)
W_u shape:   (2, 1)   (Esperado: 2x1)


In [2]:
# --- Construção das Matrizes do Sistema ---

print("--- Construindo Matrizes do Sistema de Lur'e ---")

# 1. Matrizes da Dinâmica do Estado (Linha 1 do Sistema)
# x_{k+1} = A x_k + B_w w_k + B_u u_k

# A: Matriz de Estados (2x2)
A = (1 - alpha_step) * np.eye(n_x)

# B_u: Matriz de Entrada de Controle (2x1)
B_u = alpha_step * W_u

# B_w: Matriz de Entrada da Rede Neural (2x192)
# Apenas a última camada (Output) afeta diretamente o estado x_{k+1}
# Estrutura: [0, 0, alpha * W_out]
B_w = np.block([
    np.zeros((n_x, n_h)),      # Camada 1 não afeta x direto
    np.zeros((n_x, n_h)),      # Camada 2 não afeta x direto
    alpha_step * W_out         # Camada 3 afeta x
])

# 2. Matrizes de Interconexão Interna (Linha 2 do Sistema)
# v_k = C_v x_k + D_vw w_k

# C_v: Entrada do Estado na Rede (192x2)
# Apenas a primeira camada (Input) recebe o estado x_k
# Estrutura: [W_in; 0; 0]
C_v = np.block([
    [W_in],
    [np.zeros((n_h, n_x))],
    [np.zeros((n_h, n_x))]
])

# D_vw: Matriz de Feedback Interno (192x192)
# Define como as camadas se conectam (1->2, 2->3)
# Estrutura Triangular Inferior por Blocos
D_vw = np.block([
    [np.zeros((n_h, n_h)), np.zeros((n_h, n_h)), np.zeros((n_h, n_h))], # v1 só depende de x
    [W_h1,                 np.zeros((n_h, n_h)), np.zeros((n_h, n_h))], # v2 depende de w1
    [np.zeros((n_h, n_h)), W_h2,                 np.zeros((n_h, n_h))]  # v3 depende de w2
])

# --- Validação das Dimensões Finais ---
print(f"A shape:    {A.shape}")
print(f"B_u shape:  {B_u.shape}")
print(f"B_w shape:  {B_w.shape}  (Esperado: 2x192)")
print(f"C_v shape:  {C_v.shape}  (Esperado: 192x2)")
print(f"D_vw shape: {D_vw.shape} (Esperado: 192x192)")

--- Construindo Matrizes do Sistema de Lur'e ---
A shape:    (2, 2)
B_u shape:  (2, 1)
B_w shape:  (2, 192)  (Esperado: 2x192)
C_v shape:  (192, 2)  (Esperado: 192x2)
D_vw shape: (192, 192) (Esperado: 192x192)


In [3]:
# Interval Bound Propagation (IBP) e Matrizes de Setor

print("\n--- Executando Interval Bound Propagation (IBP) ---")

# 1. Definição dos Limites do Espaço de Estados (Hyper-Rectangle)
# x_max já foi definido como [0.573, 3.0]
x_min = -x_max

print(f"Limites de Estado: \n  Min: {x_min} \n  Max: {x_max}")

# 2. Função Auxiliar para Aritmética Intervalar Linear
def propagate_affine(W, l_in, u_in, b=None):
    """
    Propaga limites [l_in, u_in] através de y = Wx + b
    Retorna [l_out, u_out]
    """
    # Decomposição em partes positiva e negativa para garantir worst-case
    W_plus = np.maximum(W, 0)
    W_minus = np.minimum(W, 0)
    
    # Limite Superior: Pesos positivos * max input + Pesos negativos * min input
    u_out = W_plus @ u_in + W_minus @ l_in
    
    # Limite Inferior: Pesos positivos * min input + Pesos negativos * max input
    l_out = W_plus @ l_in + W_minus @ u_in
    
    if b is not None:
        u_out += b
        l_out += b
        
    return l_out, u_out

# 3. Propagação Camada a Camada
# Inicialização
l_x, u_x = x_min, x_max

# --- Camada 1 (Input -> Hidden 1) ---
l_v1, u_v1 = propagate_affine(W_in, l_x, u_x) # Pré-ativação
# Ativação tanh é monotónica crescente: bounds mapeiam diretamente
l_w1, u_w1 = np.tanh(l_v1), np.tanh(u_v1)     # Pós-ativação

# --- Camada 2 (Hidden 1 -> Hidden 2) ---
l_v2, u_v2 = propagate_affine(W_h1, l_w1, u_w1)
l_w2, u_w2 = np.tanh(l_v2), np.tanh(u_v2)

# --- Camada 3 (Hidden 2 -> Hidden 3) ---
l_v3, u_v3 = propagate_affine(W_h2, l_w2, u_w2)
l_w3, u_w3 = np.tanh(l_v3), np.tanh(u_v3)

# 4. Construção do Vetor Global de Limites (nu_bar)
# Empilhamos os limites de pré-ativação de todas as camadas
# nu_bar_j = max(|lower_j|, |upper_j|)
nu_v1 = np.maximum(np.abs(l_v1), np.abs(u_v1))
nu_v2 = np.maximum(np.abs(l_v2), np.abs(u_v2))
nu_v3 = np.maximum(np.abs(l_v3), np.abs(u_v3))

nu_bar_total = np.concatenate([nu_v1, nu_v2, nu_v3])

print(f"Propagação concluída. Dimensão de nu_bar: {nu_bar_total.shape}")
print(f"  Max nu (Layer 1): {np.max(nu_v1):.4f}")
print(f"  Max nu (Layer 2): {np.max(nu_v2):.4f}")
print(f"  Max nu (Layer 3): {np.max(nu_v3):.4f}")

# 5. Cálculo das Inclinações de Setor
# alpha = tanh(nu) / nu
# Tratamento numérico para evitar divisão por zero (se nu ~ 0, alpha -> 1)
epsilon = 1e-6
alpha_vec = np.tanh(nu_bar_total) / np.maximum(nu_bar_total, epsilon)
# Se nu era muito pequeno, forçamos alpha = 1.0
alpha_vec[nu_bar_total < epsilon] = 1.0

# Matrizes Diagonais
A_alpha = np.diag(alpha_vec)
B_beta  = np.eye(N_total)

print("\nMatrizes de Setor Construídas:")
print(f"  A_alpha min slope: {np.min(alpha_vec):.4f}")
print(f"  A_alpha max slope: {np.max(alpha_vec):.4f}")


--- Executando Interval Bound Propagation (IBP) ---
Limites de Estado: 
  Min: [-0.573 -3.   ] 
  Max: [0.573 3.   ]
Propagação concluída. Dimensão de nu_bar: (192,)
  Max nu (Layer 1): 0.6861
  Max nu (Layer 2): 0.8433
  Max nu (Layer 3): 1.0773

Matrizes de Setor Construídas:
  A_alpha min slope: 0.7353
  A_alpha max slope: 0.9990


In [4]:
print(f"A_alpha shape: {A_alpha.shape} (Esperado: {N_total}x{N_total})")

A_alpha shape: (192, 192) (Esperado: 192x192)


## Viabilidade com a LMI 1 
## para H1 = 0 e H2 = $\epsilon I$


In [5]:
print("\n--- Iniciando Teste de Viabilidade (LMI 1) com MOSEK ---")

# 1. Definição das Variáveis de Decisão do Solver
# Z: Matriz de transformação cheia (2x2)
Z = cp.Variable((n_x, n_x))

# P_hat: Matriz de Lyapunov transformada, simétrica definida positiva (2x2)
P_hat = cp.Variable((n_x, n_x), symmetric=True)

# Y: Matriz de ganho projetada (1x2)
Y = cp.Variable((n_u, n_x))

# lambda_vec: Multiplicadores de Setor (Vetor 192x1, estritamente positivo)
lambda_vec = cp.Variable(N_total, nonneg=True)

# Tolerância numérica para forçar desigualdades estritas (M < -tol*I)
tol = 1e-6 

# 2. Construção dos Blocos da Matriz de Setor (Omega)
# Nota: cp.multiply faz a multiplicação elemento a elemento de vetores
Omega_11 = -2 * cp.diag(cp.multiply(alpha_vec, lambda_vec))
Omega_12 = cp.diag(cp.multiply(1 + alpha_vec, lambda_vec))
Omega_21 = Omega_12 # A matriz é diagonal, logo simétrica
Omega_22 = -2 * cp.diag(lambda_vec)

# 3. Definição do Grid de Busca para o hiperparâmetro epsilon
eps_grid = [0.01, 0.1, 1.0, 10.0]
solucao_encontrada = False

for eps in eps_grid:
    print(f"\nTestando epsilon = {eps}...")
    
    # --- Montagem dos Blocos da Matriz de Finsler ---
    
    # Linha 1 (dimensão n_x = 2)
    M11 = -P_hat
    M12 = Z.T @ A.T + Y.T @ B_u.T
    M13 = np.zeros((n_x, N_total))
    M14 = eps * Z.T @ C_v.T
    
    # Linha 2 (dimensão n_x = 2)
    M21 = A @ Z + B_u @ Y
    M22 = P_hat - Z - Z.T
    M23 = np.zeros((n_x, N_total))
    M24 = B_w
    
    # Linha 3 (dimensão N_total = 192)
    M31 = np.zeros((N_total, n_x))
    M32 = np.zeros((N_total, n_x))
    M33 = Omega_11
    M34 = Omega_12 - eps * np.eye(N_total)
    
    # Linha 4 (dimensão N_total = 192)
    M41 = eps * C_v @ Z
    M42 = B_w.T
    M43 = Omega_21 - eps * np.eye(N_total)
    M44 = Omega_22 + eps * D_vw + eps * (D_vw.T)
    
    # Empilhamento da Matriz Global (388 x 388)
    M_finsler = cp.bmat([
        [M11, M12, M13, M14],
        [M21, M22, M23, M24],
        [M31, M32, M33, M34],
        [M41, M42, M43, M44]
    ])
    
    # --- Definição das Restrições ---
    restricoes = [
        P_hat >> tol * np.eye(n_x),                     # P_hat estritamente positiva
        lambda_vec >= tol,                              # Multiplicadores estritamente positivos
        M_finsler << -tol * np.eye(2*n_x + 2*N_total)   # Finsler estritamente negativa
    ]
    
    # --- Definição do Problema de Otimização ---
    # Minimizamos 0 (Problema puramente de viabilidade)
    prob = cp.Problem(cp.Minimize(0), restricoes)
    
    # --- Resolução com MOSEK ---
    try:
        # Passamos parâmetros para evitar warnings do MOSEK sobre tolerâncias
        prob.solve(solver=cp.MOSEK, verbose=False)
        
        if prob.status == cp.OPTIMAL or prob.status == cp.OPTIMAL_INACCURATE:
            print(f" LMI 1 é VIÁVEL para epsilon = {eps}!")
            solucao_encontrada = True
            
            # Recuperação do Ganho de Controle para validação rápida
            Z_val = Z.value
            Y_val = Y.value
            K_val = Y_val @ np.linalg.inv(Z_val)
            print(f"  Ganho K encontrado: {K_val}")
            break # Interrompe a busca pois já achamos um epsilon viável
        else:
            print(f"  [FALHA] Status do solver: {prob.status}")
            
    except Exception as e:
         print(f"  [ERRO NO SOLVER] {e}")


--- Iniciando Teste de Viabilidade (LMI 1) com MOSEK ---

Testando epsilon = 0.01...
  [FALHA] Status do solver: infeasible

Testando epsilon = 0.1...
  [FALHA] Status do solver: infeasible

Testando epsilon = 1.0...
  [FALHA] Status do solver: infeasible

Testando epsilon = 10.0...
  [FALHA] Status do solver: infeasible


## Viabilidade com a LMI 1 
## para H1 = $\epsilon I$ e H2 = 0

In [6]:
print("\n--- Iniciando Teste de Viabilidade: H1 = eps*I, H2 = 0 ---")

# 1. Definição das Variáveis de Decisão do Solver
Z = cp.Variable((n_x, n_x))
P_hat = cp.Variable((n_x, n_x), symmetric=True)
Y = cp.Variable((n_u, n_x))
lambda_vec = cp.Variable(N_total, nonneg=True)

tol = 1e-6 # Tolerância numérica para desigualdades estritas

# 2. Construção dos Blocos da Matriz de Setor (Omega)
Omega_11 = -2 * cp.diag(cp.multiply(alpha_vec, lambda_vec))
Omega_12 = cp.diag(cp.multiply(1 + alpha_vec, lambda_vec))
Omega_21 = Omega_12 
Omega_22 = -2 * cp.diag(lambda_vec)

# 3. Grid Logarítmico para o Line-Search
eps_grid = [1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]
solucao_encontrada = False

for eps in eps_grid:
    print(f"Testando epsilon = {eps}...")
    
    # --- Montagem dos Blocos da Matriz de Finsler ---
    
    # Linha 1 (dimensão n_x = 2)
    M11 = -P_hat
    M12 = Z.T @ A.T + Y.T @ B_u.T
    M13 = np.zeros((n_x, N_total))
    M14 = np.zeros((n_x, N_total))
    
    # Linha 2 (dimensão n_x = 2)
    M21 = A @ Z + B_u @ Y
    M22 = P_hat - Z - Z.T
    M23 = np.zeros((n_x, N_total))
    M24 = B_w
    
    # Linha 3 (dimensão N_total = 192)
    M31 = eps * C_v @ Z
    M32 = np.zeros((N_total, n_x))
    M33 = Omega_11 - 2 * eps * np.eye(N_total)
    M34 = Omega_12 + eps * D_vw
    
    # Linha 4 (dimensão N_total = 192)
    M41 = np.zeros((N_total, n_x))
    M42 = B_w.T
    M43 = Omega_21 + eps * (D_vw.T)
    M44 = Omega_22
    
    # Empilhamento da Matriz Global
    M_finsler = cp.bmat([
        [M11, M12, M13, M14],
        [M21, M22, M23, M24],
        [M31, M32, M33, M34],
        [M41, M42, M43, M44]
    ])
    
    # --- Definição das Restrições ---
    restricoes = [
        P_hat >> tol * np.eye(n_x),
        lambda_vec >= tol,
        M_finsler << -tol * np.eye(2*n_x + 2*N_total)
    ]
    
    # --- Problema de Otimização (Viabilidade) ---
    prob = cp.Problem(cp.Minimize(0), restricoes)
    
    # --- Resolução ---
    try:
        prob.solve(solver=cp.MOSEK, verbose=False)
        
        if prob.status in [cp.OPTIMAL, cp.OPTIMAL_INACCURATE]:
            print(f"LMI 1 VIÁVEL para epsilon = {eps}!")
            solucao_encontrada = True
            
            # Validação do ganho
            K_val = Y.value @ np.linalg.inv(Z.value)
            print(f"  -> Ganho K: {K_val}")
            break # Para a busca ao achar o primeiro viável
        else:
            print(f"  -> [FALHA] Status: {prob.status}")
            
    except Exception as e:
         print(f"  -> [ERRO SOLVER] {e}")


--- Iniciando Teste de Viabilidade: H1 = eps*I, H2 = 0 ---
Testando epsilon = 0.0001...
  -> [FALHA] Status: infeasible
Testando epsilon = 0.001...
  -> [FALHA] Status: infeasible
Testando epsilon = 0.01...
  -> [FALHA] Status: infeasible
Testando epsilon = 0.1...
  -> [FALHA] Status: infeasible
Testando epsilon = 1.0...
  -> [FALHA] Status: infeasible
Testando epsilon = 10.0...
  -> [FALHA] Status: infeasible
Testando epsilon = 100.0...
  -> [FALHA] Status: infeasible
